# 第 14 章 マンデル・フレミング・モデル: 数字を変えて確かめる

**使い方**: 「Parameters」のセルの数字を書き換えて、上から順に実行します（メニューの Run → Run All Cells）。数字を変えるたびに、結果と図が変わります。

起動: アプリのディレクトリで `uv run --with jupyterlab jupyter lab notebooks/` を実行します。

記事: `docs/article/macro-basics-programming/14-mundell-fleming.md`

In [ ]:
# --- Parameters: ここの数字を書き換えます ---
mpc = 0.6              # 限界消費性向 c
b = 20                 # 投資の利子感応度 b
intercept = 400        # 為替レートによらない支出 A
n = 50                 # 為替レートへの純輸出の感応度 n
k = 0.5                # 貨幣需要の所得感応度 k
h = 100                # 貨幣需要の利子感応度 h
real_money = 250       # 実質貨幣供給 M/P（変動相場で使う）
world_rate = 2.5       # 世界の利子率 r*
exchange_rate = 1      # 為替レート e（固定相場で使う。上がると円安）
delta_government = 10  # 政府支出の増加 ΔG
delta_real_money = 10  # 実質貨幣供給の増加 Δ(M/P)
delta_exchange_rate = 0.1  # 固定相場での、為替レートの切り下げ幅

In [ ]:
# --- 準備（変更しません）: lib を読み込めるようにします ---
import sys
from pathlib import Path

_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "lib").is_dir())
if str(_root) in sys.path:
    sys.path.remove(str(_root))
sys.path.insert(0, str(_root))
_loaded_lib = sys.modules.get("lib")
if _loaded_lib is not None and (
    Path(getattr(_loaded_lib, "__file__", "") or "").resolve()
    != (_root / "lib" / "__init__.py").resolve()
):
    for _name in tuple(sys.modules):
        if _name == "lib" or _name.startswith("lib."):
            del sys.modules[_name]

## 均衡と、政策の効果

In [ ]:
from lib.chapter14.mundell_fleming import (
    SmallOpenEconomy,
    fixed_rate_devaluation_effect,
    fixed_rate_equilibrium,
    floating_rate_equilibrium,
    policy_effects_table,
)

economy = SmallOpenEconomy(
    mpc, b, intercept, n, k, h, real_money, world_rate, exchange_rate
)
print("固定相場の均衡:", fixed_rate_equilibrium(economy))
print("変動相場の均衡:", floating_rate_equilibrium(economy))
for row in policy_effects_table(economy, delta_government, delta_real_money):
    print(f"{row.regime}・{row.policy}: 所得 {row.income_change:+.2f}、"
          f"為替レート {row.exchange_rate_change:+.3f}、"
          f"実質貨幣供給 {row.real_money_change:+.2f}")
print(f"固定相場の切り下げ（{delta_exchange_rate}）の効果: "
      f"{fixed_rate_devaluation_effect(economy, delta_exchange_rate):+.2f}")

## 図: 為替制度ごとの財政政策

In [ ]:
from lib.chapter14.plot import plot_fiscal_policy_by_regime

plot_fiscal_policy_by_regime(
    economy, delta_government, income_min=600, income_max=1200
)

## 図: 4 通りの政策の効果

In [ ]:
from lib.chapter14.plot import plot_policy_effects

plot_policy_effects(economy, delta_government, delta_real_money)

## 試してみよう

- 初期設定では、固定・変動の、どちらの均衡も、所得 1000.0 です。政策の効果の表は、
  固定相場では、財政政策が +25.00（金融政策は 0.00）、変動相場では、財政政策が 0.00
  （金融政策は +20.00）です。**固定相場では財政政策、変動相場では金融政策が効きます**
- `world_rate = 3.5` にすると、固定相場の均衡の所得は 950.0、変動相場の均衡の所得は
  1200.0、為替レートは 3.0 になります。世界の利子率が上がると、固定相場では
  投資が減って所得が下がり、変動相場では、LM 曲線から所得が決まります
- `n = 100` にすると、変動相場の財政政策による為替レートの変化は −0.100 になります
  （`n = 50` では −0.200）。純輸出が為替に敏感なほど、為替レートの動きは小さくて
  すみます。所得の効果は、変わりません
- `delta_exchange_rate = 0.1` の切り下げは、固定相場で所得を +12.50 増やします